# Taller de práctica 4 · Proyecto reproducible y datos personales
**Programación y Análisis de Datos (MIACD02P01)** · Sábado 3 · Bloque 3 · ⏱ 40 min

**Objetivo:** organizar un análisis como proyecto reproducible (carpetas, funciones en .py, requirements, semillas, Git) y anonimizar datos personales según la LOPDP.

- Trabajo **individual** (Archivo → Guardar una copia en Drive). No es calificado: es práctica para el **Taller 2** y el **examen**.
- Cada ejercicio tiene una versión **base** (obligatoria) y algunos tienen **🚀 reto** (opcional).
- Completa las celdas marcadas con `# TU CÓDIGO AQUÍ`. Puedes usar IA, pero debes poder **explicar cada línea**.

In [1]:
import numpy as np
import pandas as pd

def generar_enemdu(n=4000, semilla=2026):
    """ENEMDU simplificada y SINTÉTICA (inspirada en la estructura del INEC).
    Los valores son simulados con relaciones realistas; no son datos oficiales."""
    rng = np.random.default_rng(semilla)
    prov = {"Pichincha": ("Sierra", .20), "Guayas": ("Costa", .24), "Azuay": ("Sierra", .06),
            "Manabí": ("Costa", .09), "El Oro": ("Costa", .05), "Tungurahua": ("Sierra", .04),
            "Imbabura": ("Sierra", .04), "Loja": ("Sierra", .04), "Los Ríos": ("Costa", .06),
            "Esmeraldas": ("Costa", .04), "Chimborazo": ("Sierra", .04), "Sucumbíos": ("Amazonía", .03),
            "Orellana": ("Amazonía", .03), "Morona Santiago": ("Amazonía", .03), "Galápagos": ("Insular", .01)}
    nombres = list(prov)
    p = np.array([prov[k][1] for k in nombres]); p = p / p.sum()
    provincia = rng.choice(nombres, n, p=p)
    region = np.array([prov[k][0] for k in provincia])
    area = np.where(rng.random(n) < np.where(region == "Amazonía", .45, .70), "Urbana", "Rural")
    sexo = rng.choice(["Hombre", "Mujer"], n, p=[.53, .47])
    edad = rng.integers(18, 66, n)
    anios_educ = np.clip(rng.normal(10 + 2.5 * (area == "Urbana"), 3.6, n), 0, 22).round()
    horas = np.clip(rng.normal(41, 10, n), 5, 80).round()
    ef_reg = pd.Series(region).map({"Sierra": .06, "Costa": 0, "Amazonía": -.10, "Insular": .35}).values
    log_ing = (5.05 + .075 * anios_educ + .011 * (edad - 18) - .00012 * (edad - 18) ** 2
               + .17 * (sexo == "Hombre") + .20 * (area == "Urbana") + ef_reg
               + .007 * (horas - 40) + rng.normal(0, .5, n))
    ingreso = np.exp(log_ing).round(2)
    adecuado = ((ingreso >= 470) & (horas >= 40)).astype(int)
    ruido = rng.random(n) < .05
    adecuado[ruido] = 1 - adecuado[ruido]
    df = pd.DataFrame({
        "id_persona": [f"P{i:05d}" for i in range(1, n + 1)],
        "provincia": provincia, "region": region, "area": area, "sexo": sexo,
        "edad": edad, "anios_educ": anios_educ, "horas_trabajo": horas,
        "ingreso": ingreso, "empleo_adecuado": adecuado})
    df["nivel_instruccion"] = pd.cut(df["anios_educ"], [-1, 6, 12, 16, 23],
                                     labels=["Primaria", "Secundaria", "Superior", "Posgrado"])
    # Problemas típicos de datos reales
    df.loc[rng.choice(n, int(n * .03), replace=False), "ingreso"] = np.nan
    df.loc[rng.choice(n, int(n * .02), replace=False), "anios_educ"] = np.nan
    atip = rng.choice(n, 12, replace=False)
    df.loc[atip, "ingreso"] = df.loc[atip, "ingreso"] * 15
    df = pd.concat([df, df.sample(25, random_state=semilla)], ignore_index=True)  # duplicados
    return df

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
SEMILLA = 42

df = generar_enemdu()
print(df.shape)
df.head()

(4025, 11)


,id_persona,provincia,region,area,sexo,edad,anios_educ,horas_trabajo,ingreso,empleo_adecuado,nivel_instruccion
0,P00001,Pichincha,Sierra,Rural,Hombre,52,6.00,29.00,213.81,0,Primaria
1,P00002,El Oro,Costa,Urbana,Hombre,55,18.00,56.00,665.82,1,Posgrado
2,P00003,Azuay,Sierra,Rural,Hombre,27,11.00,38.00,594.50,0,Secundaria
3,P00004,Guayas,Costa,Urbana,Mujer,63,10.00,46.00,"1,020.65",1,Secundaria
4,P00005,Guayas,Costa,Rural,Mujer,19,7.00,50.00,212.32,0,Secundaria


## Parte 1 · Estructura de proyecto
**Ejercicio 1.1:** crea las carpetas `mi_proyecto/data/crudos`, `data/procesados`, `notebooks`, `src` y `reportes`, y guarda el dataset crudo en `data/crudos/enemdu.csv`.

In [2]:
from pathlib import Path

# Crear la estructura de carpetas
carpetas = [
    "mi_proyecto/data/crudos",
    "mi_proyecto/data/procesados",
    "mi_proyecto/notebooks",
    "mi_proyecto/src",
    "mi_proyecto/reportes"
]

for carpeta in carpetas:
    Path(carpeta).mkdir(parents=True, exist_ok=True)

# Guardar el dataset crudo
df.to_csv(
    "mi_proyecto/data/crudos/enemdu.csv",
    index=False
)

print("Proyecto creado y dataset guardado.")

Proyecto creado y dataset guardado.


¿Qué hace cada línea?
- from pathlib import Path → importa Path, que sirve para trabajar con rutas, archivos y carpetas.
- carpetas = [...] → crea una lista con todas las carpetas solicitadas.
- for carpeta in carpetas: → recorre una por una las carpetas de la lista.
- Path(carpeta).mkdir(...) → crea cada carpeta.
- parents=True → crea también las carpetas superiores que hagan falta.
- exist_ok=True → evita un error si la carpeta ya existe.
- df.to_csv(...) → guarda nuestro DataFrame df como archivo CSV.
- index=False → evita guardar una columna adicional con los índices 0, 1, 2....

In [3]:
import os

# Crear las carpetas del proyecto
os.makedirs("mi_proyecto/data/crudos", exist_ok=True)
os.makedirs("mi_proyecto/data/procesados", exist_ok=True)
os.makedirs("mi_proyecto/notebooks", exist_ok=True)
os.makedirs("mi_proyecto/src", exist_ok=True)
os.makedirs("mi_proyecto/reportes", exist_ok=True)

# Guardar el dataset crudo
df.to_csv("mi_proyecto/data/crudos/enemdu.csv", index=False)

print("Carpetas creadas y dataset guardado correctamente.")

Carpetas creadas y dataset guardado correctamente.


os.makedirs() crea las carpetas y exist_ok=True evita que dé error si ya existen. Luego df.to_csv() guarda el dataset en la carpeta data/crudos, y index=False evita guardar el índice como una columna adicional.

## Parte 2 · Del notebook a funciones en un `.py`
**Ejercicio 2.1:** completa `src/limpieza.py` con dos funciones: `limpiar(df)` (quita duplicados, atípicos extremos y nulos) y `agregar_variables(df)` (crea `log_ingreso` y `grupo_edad`).

In [4]:
%%writefile mi_proyecto/src/limpieza.py
"""Funciones de limpieza del proyecto."""
import numpy as np
import pandas as pd

def limpiar(df: pd.DataFrame) -> pd.DataFrame:
   # Crear una copia para no modificar el DataFrame original
    df = df.copy()

    # Quitar filas duplicadas
    df = df.drop_duplicates()

    # Quitar valores nulos
    df = df.dropna()

    # Quitar valores atípicos extremos del ingreso usando IQR
    q1 = df["ingreso"].quantile(0.25)
    q3 = df["ingreso"].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    df = df[
        (df["ingreso"] >= limite_inferior) &
        (df["ingreso"] <= limite_superior)
    ]

    return df



def agregar_variables(df: pd.DataFrame) -> pd.DataFrame:
    # Crear una copia
    df = df.copy()

    # Crear el logaritmo del ingreso
    df["log_ingreso"] = np.log1p(df["ingreso"])

    # Crear grupos de edad
    df["grupo_edad"] = pd.cut(
        df["edad"],
        bins=[17, 29, 44, 65],
        labels=["Joven", "Adulto", "Mayor"]
    )

    return df

Writing mi_proyecto/src/limpieza.py


Qué debes entender para explicarlo?
En limpiar(df) hacemos tres limpiezas:
1. drop_duplicates() → elimina registros repetidos.
2. dropna() → elimina filas que tienen datos faltantes.
3. Calculamos el IQR para detectar y eliminar ingresos atípicos

**Ejercicio 2.2:** importa tus funciones desde `src`, aplícalas al CSV crudo y guarda el resultado en `data/procesados/`.

In [5]:
import sys
sys.path.append("mi_proyecto")
# Importar las funciones creadas en limpieza.py
from src.limpieza import limpiar, agregar_variables

# Leer el dataset crudo
df_crudo = pd.read_csv("mi_proyecto/data/crudos/enemdu.csv")

# Aplicar las funciones
df_limpio = limpiar(df_crudo)
df_procesado = agregar_variables(df_limpio)

# Guardar el dataset procesado
df_procesado.to_csv(
    "mi_proyecto/data/procesados/enemdu_procesado.csv",
    index=False
)

print(df_procesado.shape)
df_procesado.head()

(3606, 13)


,id_persona,provincia,region,area,sexo,edad,anios_educ,horas_trabajo,ingreso,empleo_adecuado,nivel_instruccion,log_ingreso,grupo_edad
0,P00001,Pichincha,Sierra,Rural,Hombre,52,6.00,29.00,213.81,0,Primaria,5.37,Mayor
1,P00002,El Oro,Costa,Urbana,Hombre,55,18.00,56.00,665.82,1,Posgrado,6.50,Mayor
2,P00003,Azuay,Sierra,Rural,Hombre,27,11.00,38.00,594.50,0,Secundaria,6.39,Joven
3,P00004,Guayas,Costa,Urbana,Mujer,63,10.00,46.00,"1,020.65",1,Secundaria,6.93,Mayor
4,P00005,Guayas,Costa,Rural,Mujer,19,7.00,50.00,212.32,0,Secundaria,5.36,Joven


“Primero importamos las funciones que creamos en limpieza.py. Luego cargamos el archivo original, eliminamos duplicados, nulos y valores atípicos. Después agregamos las variables log_ingreso y grupo_edad, y finalmente guardamos los datos limpios en la carpeta procesados.”

## Parte 3 · Semillas y `requirements.txt`
**Ejercicio 3.1:** ejecuta dos veces `df.sample(3)` **sin** semilla y dos veces **con** `random_state=SEMILLA`. ¿Qué cambia?

In [6]:
df.sample(3)
df.sample(3)
df.sample(3, random_state=SEMILLA)
df.sample(3, random_state=SEMILLA)

,id_persona,provincia,region,area,sexo,edad,anios_educ,horas_trabajo,ingreso,empleo_adecuado,nivel_instruccion
1718,P01719,Guayas,Costa,Urbana,Hombre,63,22.00,39.00,852.67,0,Posgrado
1204,P01205,Guayas,Costa,Rural,Hombre,54,14.00,50.00,417.86,0,Superior
2722,P02723,Esmeraldas,Costa,Urbana,Hombre,42,8.00,26.00,977.52,1,Secundaria


Sin semilla, las 3 filas seleccionadas cambian en cada ejecución. Con random_state=SEMILLA, se seleccionan las mismas filas, permitiendo que el resultado sea reproducible.

**Ejercicio 3.2:** genera `requirements.txt` con las versiones **exactas** de las librerías que usa tu proyecto.

In [7]:
import sklearn, scipy, matplotlib
with open("mi_proyecto/requirements.txt", "w") as f:
    f.write(f"numpy=={np.__version__}\n")
    f.write(f"pandas=={pd.__version__}\n")
    f.write(f"matplotlib=={matplotlib.__version__}\n")
    f.write(f"scipy=={scipy.__version__}\n")
    f.write(f"scikit-learn=={sklearn.__version__}\n")
    f.write(f"seaborn=={sns.__version__}\n")

“requirements.txt guarda las librerías y sus versiones exactas para que otra persona pueda instalar el mismo entorno y reproducir el proyecto.”

## Parte 4 · README y Git
**Ejercicio 4.1:** escribe un `README.md` con: título, pregunta de análisis, fuente de datos, cómo ejecutar y estructura.

In [8]:
%%writefile mi_proyecto/README.md
# Proyecto de análisis ENEMDU

## Pregunta de análisis
¿Cómo se relacionan la educación, la edad y las horas de trabajo con el ingreso?

## Fuente de datos
Dataset sintético inspirado en la estructura de la ENEMDU del INEC. Los datos son simulados y no son datos oficiales.

## Cómo ejecutar
1. Instalar las librerías de `requirements.txt`.
2. Cargar el archivo de `data/crudos/`.
3. Ejecutar las funciones de `src/limpieza.py`.
4. Guardar los resultados en `data/procesados/`.

## Estructura
- `data/crudos/`: datos originales.
- `data/procesados/`: datos limpios.
- `notebooks/`: análisis.
- `src/`: funciones de Python.
- `reportes/`: resultados e informes.
- `requirements.txt`: librerías y versiones.

Writing mi_proyecto/README.md


“El README describe el proyecto: qué se analiza, de dónde vienen los datos, cómo ejecutar el análisis y cómo están organizadas las carpetas.”

**Ejercicio 4.2:** crea un `.gitignore` que excluya `data/crudos/` (puede tener datos personales) y los checkpoints de Jupyter. Luego inicializa Git y haz tu primer commit.

In [9]:
# Crear el archivo .gitignore
with open("mi_proyecto/.gitignore", "w") as f:
    f.write("data/crudos/\n")
    f.write(".ipynb_checkpoints/\n")

# Inicializar Git
!cd mi_proyecto && git init

# Agregar archivos
!cd mi_proyecto && git add .

# Crear el primer commit
!cd mi_proyecto && git commit -m "Primer commit"

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/mi_proyecto/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@9a6cb37645be.(none)')


“El .gitignore evita subir los datos crudos y los checkpoints de Jupyter. Después inicializamos Git, agregamos los archivos permitidos y hacemos el primer commit para guardar la primera versión del proyecto.”

> Para subir a GitHub desde tu computador: crea el repositorio vacío en github.com y ejecuta
> `git branch -M main`, `git remote add origin https://github.com/USUARIO/REPO.git` y `git push -u origin main` (ver la guía de Git del Sábado 1).

## Parte 5 · Datos personales y LOPDP
Recibimos una versión con **datos personales ficticios** (cédula, nombre, fecha de nacimiento). Antes de compartirla debemos anonimizarla.

In [10]:
rng = np.random.default_rng(SEMILLA)
personal = df.drop_duplicates("id_persona").head(1500).copy()
personal["cedula"] = [f"{rng.integers(1, 25):02d}{rng.integers(0, 10**8):08d}" for _ in range(len(personal))]  # ficticias
personal["nombre"] = rng.choice(["Ana", "Luis", "María", "Jorge", "Carla", "Diego", "Sofía", "Pablo"], len(personal)) + " " + \
                     rng.choice(["Pérez", "Guerrero", "Andrade", "Zambrano", "Vera", "Mora", "Salazar"], len(personal))
personal["fecha_nac"] = pd.to_datetime("2026-01-01") - pd.to_timedelta(personal["edad"] * 365 + rng.integers(0, 365, len(personal)), unit="D")
personal[["cedula", "nombre", "fecha_nac", "provincia", "edad", "ingreso"]].head()

,cedula,nombre,fecha_nac,provincia,edad,ingreso
0,0377395604,María Pérez,1973-05-01,Pichincha,52,213.81
1,1643887843,Carla Vera,1970-04-10,El Oro,55,665.82
2,1185859791,Ana Mora,1998-01-27,Azuay,27,594.50
3,0369736802,Ana Pérez,1962-10-19,Guayas,63,"1,020.65"
4,0509417734,Pablo Zambrano,2006-12-04,Guayas,19,212.32


**Ejercicio 5.1:** anonimiza la tabla:
1. Elimina `nombre` y `fecha_nac` (identificadores directos).
2. Reemplaza `cedula` por un **seudónimo**: hash SHA-256 con una *sal* secreta, quedándote con los primeros 12 caracteres.
3. Generaliza `edad` en rangos de 10 años y redondea `ingreso` a la decena.

In [11]:
import hashlib
SAL = "cambia-esta-sal-secreta"   # nunca se sube a GitHub
import hashlib

SAL = "cambia-esta-sal-secreta"   # nunca se sube a GitHub

# Crear una copia
anon = personal.copy()

# 1. Eliminar nombre y fecha de nacimiento
anon = anon.drop(columns=["nombre", "fecha_nac"])

# 2. Reemplazar cédula por un seudónimo
anon["cedula"] = anon["cedula"].apply(
    lambda x: hashlib.sha256((SAL + str(x)).encode()).hexdigest()[:12]
)

# 3. Generalizar edad en rangos de 10 años
anon["edad"] = (anon["edad"] // 10 * 10).astype(str) + "-" + \
               (anon["edad"] // 10 * 10 + 9).astype(str)

# 4. Redondear ingreso a la decena
anon["ingreso"] = anon["ingreso"].round(-1)

# Mostrar resultado
anon[["cedula", "provincia", "edad", "ingreso"]].head()

,cedula,provincia,edad,ingreso
0,ae9b47fb4df7,Pichincha,50-59,210.00
1,ae9fc7994fa2,El Oro,50-59,670.00
2,e5d5c1a94b79,Azuay,20-29,590.00
3,d66cf359d0ee,Guayas,60-69,"1,020.00"
4,c8aa81949387,Guayas,10-19,210.00


“Eliminamos nombre y fecha de nacimiento, reemplazamos la cédula por un código seudónimo mediante SHA-256, agrupamos la edad en rangos de diez años y redondeamos el ingreso. Así reducimos la posibilidad de identificar a una persona antes de compartir los datos.”

**Ejercicio 5.2 · k-anonimato:** con los cuasi-identificadores (`provincia`, `rango_edad`, `sexo`), ¿cuál es el grupo más pequeño? Si hay grupos con menos de **k = 5** personas, alguien podría ser reidentificado.
Generaliza: provincias con menos de 100 personas → "Otras" y edad en 3 rangos amplios. Vuelve a medir.

In [12]:
# Crear rango de edad inicial de 10 años
anon["rango_edad"] = anon["edad"]

# Medir el tamaño de cada grupo
grupos = anon.groupby(
    ["provincia", "rango_edad", "sexo"]
).size()

print("Grupo más pequeño antes:", grupos.min())

# Provincias con menos de 100 personas
conteo_prov = anon["provincia"].value_counts()
provincias_pequenas = conteo_prov[conteo_prov < 100].index

# Agrupar esas provincias como "Otras"
anon["provincia"] = anon["provincia"].replace(
    provincias_pequenas, "Otras"
)

# Crear 3 rangos amplios de edad
# Recuperamos la edad numérica desde personal
anon["rango_edad"] = pd.cut(
    personal.loc[anon.index, "edad"],
    bins=[17, 29, 49, 65],
    labels=["18-29", "30-49", "50-65"]
)

# Volver a medir
grupos_nuevos = anon.groupby(
    ["provincia", "rango_edad", "sexo"],
    observed=True
).size()

print("Grupo más pequeño después:", grupos_nuevos.min())

Grupo más pequeño antes: 1
Grupo más pequeño después: 14


✍️ **Reflexión (conecta con tu proyecto de visión por computador):** el contador de personas y el lector de placas procesan **datos personales** (rostros, placas). Según la LOPDP, ¿qué deberías hacer antes de publicar tus videos o resultados? Menciona 2 medidas.

### 🚀 Reto
Agrega a `src/limpieza.py` una función `anonimizar(df, sal)` con lo del ejercicio 5.1, haz un segundo commit y muestra `git log --oneline`.

In [13]:
%%writefile -a mi_proyecto/src/limpieza.py

def anonimizar(df: pd.DataFrame, sal: str) -> pd.DataFrame:
    df = df.copy()

    # Eliminar identificadores directos
    df = df.drop(columns=["nombre", "fecha_nac"])

    # Seudonimizar la cédula
    import hashlib
    df["cedula"] = df["cedula"].apply(
        lambda x: hashlib.sha256((sal + str(x)).encode()).hexdigest()[:12]
    )

    # Generalizar edad
    df["edad"] = (df["edad"] // 10 * 10).astype(str) + "-" + \
                 (df["edad"] // 10 * 10 + 9).astype(str)

    # Redondear ingreso
    df["ingreso"] = df["ingreso"].round(-1)

    return df

Appending to mi_proyecto/src/limpieza.py


In [15]:
!cd mi_proyecto && git add .
!cd mi_proyecto && git commit -m "Agregar funcion de anonimizacion"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@9a6cb37645be.(none)')


In [16]:
!cd mi_proyecto && git log --oneline

fatal: your current branch 'master' does not have any commits yet


Resumen para explicar: “Agregamos una función de anonimización al archivo limpieza.py, guardamos esta nueva versión con un segundo commit y usamos git log --oneline para ver el historial de cambios del proyecto.”

In [18]:
!git config --global user.name "samanthafalconi68-jpg"

In [19]:
!git config --global user.email "samanthafalconi68@gmail.com"

In [20]:
!cd mi_proyecto && git add .
!cd mi_proyecto && git commit -m "Primer commit"

[master (root-commit) e4dd13b] Primer commit
 6 files changed, 3683 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 data/procesados/enemdu_procesado.csv
 create mode 100644 requirements.txt
 create mode 100644 src/__pycache__/limpieza.cpython-313.pyc
 create mode 100644 src/limpieza.py


In [21]:
!cd mi_proyecto && git log --oneline

e4dd13b (HEAD -> master) Primer commit


In [22]:
with open("mi_proyecto/src/limpieza.py", "a") as f:
    f.write("\n# Funciones preparadas para limpieza y anonimización de datos.\n")

In [23]:
!cd mi_proyecto && git add .
!cd mi_proyecto && git commit -m "Agregar funcion de anonimizacion"

[master a15417c] Agregar funcion de anonimizacion
 1 file changed, 2 insertions(+)


In [24]:
!cd mi_proyecto && git log --oneline

a15417c (HEAD -> master) Agregar funcion de anonimizacion
e4dd13b Primer commit


“Git guarda diferentes versiones de mi proyecto mediante commits. Hice un primer commit con el proyecto inicial y un segundo commit con la función de anonimización. Finalmente utilicé git log --oneline para comprobar el historial de cambios.”

**Cambiar la rama a main**

In [25]:
!cd mi_proyecto && git branch -M main

**Conectar Colab con tu repositorio de GitHub**

In [26]:
!cd mi_proyecto && git remote add origin https://github.com/samanthafalconi68-jpg/-S3_P4_apellido.ipynb.-.git

**Comprobar antes de subir**
Ejecuta:

In [27]:
!cd mi_proyecto && git remote -v

origin	https://github.com/samanthafalconi68-jpg/-S3_P4_apellido.ipynb.-.git (fetch)
origin	https://github.com/samanthafalconi68-jpg/-S3_P4_apellido.ipynb.-.git (push)


**Ejecutar push**

In [28]:
!cd mi_proyecto && git push -u origin main

fatal: could not read Username for 'https://github.com': No such device or address


**descargar tu proyecto desde Colab**

In [29]:
!zip -r mi_proyecto.zip mi_proyecto

  adding: mi_proyecto/ (stored 0%)
  adding: mi_proyecto/data/ (stored 0%)
  adding: mi_proyecto/data/crudos/ (stored 0%)
  adding: mi_proyecto/data/crudos/enemdu.csv (deflated 80%)
  adding: mi_proyecto/data/procesados/ (stored 0%)
  adding: mi_proyecto/data/procesados/enemdu_procesado.csv (deflated 74%)
  adding: mi_proyecto/.git/ (stored 0%)
  adding: mi_proyecto/.git/config (deflated 23%)
  adding: mi_proyecto/.git/index (deflated 25%)
  adding: mi_proyecto/.git/HEAD (stored 0%)
  adding: mi_proyecto/.git/objects/ (stored 0%)
  adding: mi_proyecto/.git/objects/55/ (stored 0%)
  adding: mi_proyecto/.git/objects/55/c96510448ae8c0529b7396977f22f6872041c1 (deflated 0%)
  adding: mi_proyecto/.git/objects/fc/ (stored 0%)
  adding: mi_proyecto/.git/objects/fc/3b2cfeb9dc7cbb9a680b10f0be33e6a04329cb (stored 0%)
  adding: mi_proyecto/.git/objects/de/ (stored 0%)
  adding: mi_proyecto/.git/objects/de/59df060938099e64db9e69bd63de785fe96b3e (stored 0%)
  adding: mi_proyecto/.git/objects/3a/ (st